In [1]:
import os
import shutil
import random

# --- Configuration ---
source_img_dir = "Annotated_Images/train/images"
source_lbl_dir = "Annotated_Images/train/labels" 
base_out_dir = "YOLO_Dataset"

# Splitting ratios
train_ratio, val_ratio, test_ratio = 0.7, 0.2, 0.1

# Create new directory structure
dirs_to_make = [
    f"{base_out_dir}/images/train", f"{base_out_dir}/labels/train",
    f"{base_out_dir}/images/val", f"{base_out_dir}/labels/val",
    f"{base_out_dir}/images/test", f"{base_out_dir}/labels/test"
]
for d in dirs_to_make:
    os.makedirs(d, exist_ok=True)

# Get all images and labels
images = [f for f in os.listdir(source_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
data_pairs = []

for img_name in images:
    lbl_name = os.path.splitext(img_name)[0] + ".txt"
    if os.path.exists(os.path.join(source_lbl_dir, lbl_name)):
        data_pairs.append((img_name, lbl_name))

print(f"Found {len(data_pairs)} matching image-label pairs.")

# Shuffle and split
random.seed(42)
random.shuffle(data_pairs)

total = len(data_pairs)
train_end = int(total * train_ratio)
val_end = train_end + int(total * val_ratio)

train_pairs = data_pairs[:train_end]
val_pairs = data_pairs[train_end:val_end]
test_pairs = data_pairs[val_end:]

def copy_files(pairs, split_name):
    for img_name, lbl_name in pairs:
        shutil.copy(os.path.join(source_img_dir, img_name), os.path.join(base_out_dir, f"images/{split_name}", img_name))
        shutil.copy(os.path.join(source_lbl_dir, lbl_name), os.path.join(base_out_dir, f"labels/{split_name}", lbl_name))
    print(f"Copied {len(pairs)} files to {split_name}.")

copy_files(train_pairs, "train")
copy_files(val_pairs, "val")
copy_files(test_pairs, "test")

Found 1201 matching image-label pairs.
Copied 840 files to train.
Copied 240 files to val.
Copied 121 files to test.


In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import albumentations as A
import cv2
import os

# --- Configuration ---
train_img_dir = "YOLO_Dataset/images/train"
train_lbl_dir = "YOLO_Dataset/labels/train"
augment_multiplier = 3  # Increased from 2 to get more variety from the richer pipeline

transform = A.Compose([
    # --- Geometry ---
    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.04,
        scale_limit=0.08,
        rotate_limit=12,
        border_mode=0,
        p=0.4
    ),

    A.RandomScale(scale_limit=0.12, p=0.3),

    A.Perspective(
        scale=(0.01, 0.04),
        p=0.2
    ),

    # --- Blur / Focus / Motion ---
    A.MotionBlur(blur_limit=(3, 7), p=0.25),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),

    # --- Lighting, safer than hue/brightness extreme ---
    A.CLAHE(
        clip_limit=2.0,
        tile_grid_size=(8, 8),
        p=0.15
    ),

    A.RandomShadow(
        shadow_roi=(0, 0, 1, 1),
        num_shadows_lower=1,
        num_shadows_upper=2,
        shadow_dimension=4,
        p=0.15
    ),

    # --- Camera / Video Quality ---
    A.ImageCompression(
        quality_lower=70,
        quality_upper=100,
        p=0.3
    ),

    A.GaussNoise(
        var_limit=(5.0, 20.0),
        p=0.2
    ),

], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.2
))

images = [f for f in os.listdir(train_img_dir) 
          if f.endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('aug_')]
total_augmented = 0

for img_name in images:
    img_path = os.path.join(train_img_dir, img_name)
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(img_name)[0] + ".txt")
    
    # Read image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Read labels
    bboxes = []
    class_labels = []
    with open(lbl_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) == 5:
                class_labels.append(int(float(parts[0])))
                bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])
                
    # Generate augmented copies
    for i in range(augment_multiplier):
        try:
            augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            aug_img = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_labels = augmented['class_labels']
            
            # Save augmented image
            aug_img_name = f"aug_{i}_{img_name}"
            aug_img_path = os.path.join(train_img_dir, aug_img_name)
            cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            
            # Save augmented labels
            aug_lbl_name = os.path.splitext(aug_img_name)[0] + ".txt"
            aug_lbl_path = os.path.join(train_lbl_dir, aug_lbl_name)
            with open(aug_lbl_path, 'w') as f:
                for bbox, label in zip(aug_bboxes, aug_labels):
                    f.write(f"{label} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            total_augmented += 1
        except Exception as e:
            pass

print(f"Augmentation complete. Generated {total_augmented} new training images.")

c:\Users\bombo\anaconda3\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
c:\Users\bombo\anaconda3\Lib\site-packages\albumentations\core\validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\Axioo Pongo\AppData\Local\Temp\ipykernel_24400\927959895.py:43: UserWarning: Argument(s) 'num_shadows_lower, num_shadows_upper' are not valid for transform RandomShadow
  A.RandomShadow(
C:\Users\Axioo Pongo\AppData\Local\Temp\ipykernel_24400\927959895.py:52: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(
C:\Users\Axioo Pongo\AppData\Local\Temp\ipykernel_2440

Augmentation complete. Generated 2520 new training images.


In [3]:
# import yaml
# from ultralytics import YOLO
# import os

# # 1. Create the dataset.yaml file required by YOLO
# yaml_content = {
#     'path': os.path.abspath('YOLO_Dataset'),
#     'train': 'images/train',
#     'val': 'images/val',
#     'test': 'images/test',
#     'names': {
#         0: '1-ball',
#         1: '2-ball',
#         2: '3-ball',
#         3: '4-ball',
#         4: '5-ball',
#         5: '6-ball',
#         6: '7-ball',
#         7: '8-ball',
#         8: '9-ball',
#         9: 'cue',
#         10: 'cue_stick',
#         11: 'pocket',
#         12: 'rack'
#     }
# }

# with open('dataset.yaml', 'w') as f:
#     yaml.dump(yaml_content, f, sort_keys=False)

# print("dataset.yaml created successfully.")

# # 2. Download YOLOv11n and Train
# model = YOLO('yolo11n.pt') 

# print("Starting training...")
# results = model.train(
#     data='dataset.yaml',
#     epochs=100,
#     imgsz=640,
#     batch=8,
#     patience=20,     # Early stopping
#     workers=4,
#     close_mosaic=20,
#     cos_lr=True,
#     project='Pool_Detection',
#     name='yolo11n_run'
# )

In [4]:
# from ultralytics import YOLO

# model = YOLO("")